# 5. Model Evaluation

Accuracy is limited for rare fraud; interpret precision, recall, Average Precision, ROC-AUC and confusion counts together.

## Corrected holdout results

The duplicate-aware holdout contains 56,948 rows and 96 frauds (0.168576%). At the fixed 0.5 threshold:

| Model | Accuracy | Precision | Recall | F1 | Average Precision | ROC-AUC | TP | FP | FN | TN |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.976364 | 0.059238 | 0.875000 | 0.110964 | 0.674710 | 0.966023 | 84 | 1,334 | 12 | 55,518 |
| Random Forest | 0.999473 | 0.971429 | 0.708333 | 0.819277 | 0.808699 | 0.940928 | 68 | 2 | 28 | 56,850 |

Logistic Regression catches more fraud but creates substantially more false alarms. Random Forest produces fewer false alarms while missing more fraud. The prior row-random results are in `sensitivity_comparison.csv`; the new grouped results are primary because duplicate feature rows do not cross the final split.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
if not (ROOT/"outputs"/"model_comparison.csv").exists():
    raise FileNotFoundError("Run notebook 04 or `python -m src.train` first to generate evaluation outputs.")

In [ ]:
import pandas as pd
comparison=pd.read_csv(ROOT/"outputs"/"model_comparison.csv")
thresholds=pd.read_csv(ROOT/"outputs"/"threshold_analysis.csv")
display(comparison); display(thresholds)

Threshold comparisons are calculated on the validation subset of training data. No winning threshold is selected because business review capacity and relative costs were not supplied. Final test metrics use the fixed 0.5 baseline threshold, once, on the untouched duplicate-aware holdout. True positives are detected fraud; false positives are legitimate cases flagged; false negatives are missed fraud; true negatives are correctly cleared.

In [ ]:
from sklearn.metrics import precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
scored = pd.read_csv(ROOT / "outputs" / "scored_transactions.csv")
models = list(scored.model.unique())
fig, axes = plt.subplots(1, 1 + len(models), figsize=(6 * (1 + len(models)), 4))
for name, group in scored.groupby("model"):
    precision, recall, _ = precision_recall_curve(group.actual_class, group.fraud_probability)
    axes[0].plot(recall, precision, label=name)
axes[0].set(title="Precision-recall curves (holdout)", xlabel="Recall", ylabel="Precision")
axes[0].legend()
for ax, name in zip(axes[1:], models):
    group = scored[scored.model == name]
    ConfusionMatrixDisplay(confusion_matrix(group.actual_class, group.predicted_label, labels=[0, 1]), display_labels=["Legitimate", "Fraud"]).plot(ax=ax, colorbar=False)
    ax.set_title("Threshold 0.5: " + name)
plt.tight_layout(); plt.show()